# Quantum Approximate Optimisation Algorithm (QAOA)

## QAOA (Farhi, Goldstone, Gutmann 2014) is a variational algorithm for combinatorial optimisation problems. Like VQE, it uses a parameterised quantum circuit with classical optimisation of the parameters — but with a particular structured ansatz built from the problem.

# 1. The problem class

## QAOA targets **MaxCut**, **Max-$k$-SAT**, and other constraint-satisfaction problems. Encode the cost function as

$$ \Large  C(z) = \sum_\alpha C_\alpha(z), \qquad z \in \{0, 1\}^n, $$

## and define the diagonal Hamiltonian $H_C = \sum_z C(z) |z\rangle\langle z|$. The goal: find $z^\star$ maximising $C$, or equivalently the ground state of $-H_C$.

# 2. The QAOA ansatz

## Define a *mixer* Hamiltonian $H_M = \sum_i X_i$. The QAOA-$p$ ansatz alternates:

$$ \Large  |\psi(\vec\gamma, \vec\beta)\rangle = e^{-i\beta_p H_M}\, e^{-i\gamma_p H_C}\, \cdots\, e^{-i\beta_1 H_M}\, e^{-i\gamma_1 H_C}\, |+\rangle^{\otimes n}. $$

## Each $e^{-i\gamma H_C}$ is implementable as a small circuit (since $H_C$ is diagonal, it is just a product of phase gates). Each $e^{-i\beta H_M}$ is just $X$-rotations on every qubit.

## Total: $2p$ parameters.

# 3. The optimisation

## Classically optimise $\vec\gamma, \vec\beta$ to maximise $\langle H_C \rangle$. After many shots and many iterations, sample $z$ from $|\psi(\vec\gamma^\star, \vec\beta^\star)\rangle$ — with high probability the sampled $z$ has $C(z)$ close to the optimum.

## As $p \to \infty$, QAOA recovers the **adiabatic algorithm** for optimisation; performance increases with $p$, but so does the circuit depth.

In [1]:
# A small QAOA-1 for MaxCut on a triangle (3 vertices, 3 edges).
import numpy as np
from scipy.optimize import minimize

edges = [(0,1), (1,2), (0,2)]
n = 3

Z = np.array([[1,0],[0,-1]]); I = np.eye(2)
def Zi(i):
    op = 1
    for k in range(n):
        op = np.kron(op, Z if k == i else I)
    return op
X = np.array([[0,1],[1,0]])
def Xi(i):
    op = 1
    for k in range(n):
        op = np.kron(op, X if k == i else I)
    return op

H_C = sum(0.5*(np.eye(2**n) - Zi(i) @ Zi(j)) for (i,j) in edges)  # number of cut edges
H_M = sum(Xi(i) for i in range(n))

from scipy.linalg import expm
def qaoa_state(params):
    gamma, beta = params
    psi = np.ones(2**n) / np.sqrt(2**n)
    psi = expm(-1j*gamma*H_C) @ psi
    psi = expm(-1j*beta *H_M) @ psi
    return psi

def expected_cut(params):
    psi = qaoa_state(params)
    return -(psi.conj() @ H_C @ psi).real  # minimise -<H_C>

res = minimize(expected_cut, x0=[0.7, 0.3], method='COBYLA')
print(f'QAOA-1 expected cut size: {-res.fun:.4f}')
print(f'Max possible cut size  : 2  (the triangle is 3 edges, max cut = 2)')

QAOA-1 expected cut size: 2.0000
Max possible cut size  : 2  (the triangle is 3 edges, max cut = 2)


# 4. Theoretical guarantees and limits

## - **MaxCut on 3-regular graphs**: QAOA-1 already beats the trivial 0.5 approximation, achieving 0.6924.
## - Goemans–Williamson, a classical algorithm, gives 0.878 for MaxCut. To beat it, QAOA needs deeper circuits and there are   no known unconditional guarantees yet.
## - Recent results (Farhi–Harrow + others) show that QAOA at constant $p$ cannot beat known classical algorithms for some   graph problems; the empirical evidence is mixed.
## - **Hardware reality**: depth $p$ is limited by decoherence; on noisy devices, $p = 1$ or $2$ are typical.

# 5. The bigger picture

## QAOA is part of a broader family of variational quantum algorithms (VQA) that try to use shallow quantum circuits as expressive ansatzes for optimisation. Whether they will offer genuine quantum advantage is one of the most-debated questions in near-term quantum computing — and very much an open problem.

# Recap

## - Variational algorithm for combinatorial optimisation.
## - Alternates problem Hamiltonian $H_C$ and mixer $H_M$ for $p$ rounds.
## - $p \to \infty$ recovers adiabatic; small $p$ is hardware-friendly.
## - Quantum advantage still an open question.